In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import duckdb

In [ ]:
df_inicial = pd.read_parquet('qtde_pessoas_familias_cadunico.parquet')

In [ ]:
df_inicial.head()

In [ ]:
df_qtde_por_ano = (
    df_inicial.query('''
        mes == 12
    ''')
    .melt(
        id_vars='ano',
        value_vars=['qtde_pessoas', 'qtde_familias'],
        var_name='grupo',
        value_name='qtde'
    )
    .replace({
        'qtde_pessoas': 'Pessoas',
        'qtde_familias': 'Famílias'
    })
    .assign(grupo = lambda x: pd.Categorical(
        x['grupo'], 
        categories=['Pessoas', 'Famílias'])
    )
    .groupby(['ano', 'grupo'])
    .agg('sum')
    .reset_index()
)

In [ ]:
df_qtde_por_ano.head()

In [ ]:
df_qtde_por_ano.dtypes

In [ ]:
df_qtde_por_ano['qtde'] = pd.to_numeric(df_qtde_por_ano['qtde'], downcast='integer')

In [ ]:
df_qtde_por_ano['qtde_1e6'] = df_qtde_por_ano['qtde'] / 1e6

In [ ]:
df_qtde_por_ano.dtypes

In [ ]:
anos = sorted(df_inicial['ano'].unique().tolist())
print(anos)

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 3.5),
    dpi=300,
    layout='constrained'
)

sns.lineplot(
    df_qtde_por_ano,
    x='ano',
    y='qtde_1e6',
    hue='grupo',
    style='grupo',
    markers=['o', '>'],
    errorbar=None,
    dashes=False,
    ax=ax
)

ax.spines[['top', 'right']].set_visible(False)
ax.set_ylabel('Quantidade (Milhões)')
ax.set_xlabel('Ano')
ax.set_xticks(anos, labels=anos, rotation=90)
ax.grid(color='grey', linestyle='--', alpha=0.5)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1), ncols=2, frameon=False)

fig.savefig('pessoas_familias_cadunico', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_qtde_por_ano_regiao = (
    df_inicial.query('''
        mes == 12
    ''')
    .melt(
        id_vars=['ano', 'regiao'],
        value_vars=['qtde_pessoas', 'qtde_familias'],
        var_name='grupo',
        value_name='qtde'
    )
    .replace({
        'qtde_pessoas': 'Pessoas',
        'qtde_familias': 'Famílias'
    })
    .assign(grupo = lambda x: pd.Categorical(
        x['grupo'], 
        categories=['Pessoas', 'Famílias'])
    )
    .groupby(['ano', 'grupo', 'regiao'])
    .agg('sum')
    .reset_index()
    .assign(
        qtde = lambda x: pd.to_numeric(
            x['qtde'], 
            downcast='integer'
        ),
        qtde_1e6 = lambda x: x['qtde'] / 1e6
    )
)

In [ ]:
df_qtde_por_ano_regiao.head()

In [ ]:
df_qtde_por_ano_regiao.dtypes

In [ ]:
for (regiao, df_por_regiao) in df_qtde_por_ano_regiao.groupby('regiao'):
    print(regiao)
    print(type(df_por_regiao))

    fig, ax = plt.subplots(
        figsize=(10, 3.5),
        dpi=300,
        layout='constrained'
    )

    sns.lineplot(
        df_por_regiao,
        x='ano',
        y='qtde_1e6',
        hue='grupo',
        style='grupo',
        markers=['o', '>'],
        errorbar=None,
        dashes=False,
        ax=ax
    )

    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylabel('Quantidade (Milhões)')
    ax.set_xlabel('Ano')
    ax.set_xticks(anos, labels=anos, rotation=90)
    ax.grid(color='grey', linestyle='--', alpha=0.5)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1), ncols=2, frameon=False)

    fig.savefig(f'pessoas_familias_cadunico_{regiao.lower()}', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
df_qtde_por_ano_regiao_uf = (
    df_inicial.query('''
        mes == 12
    ''')
    .melt(
        id_vars=['ano', 'regiao', 'sigla_uf'],
        value_vars=['qtde_pessoas', 'qtde_familias'],
        var_name='grupo',
        value_name='qtde'
    )
    .replace({
        'qtde_pessoas': 'Pessoas',
        'qtde_familias': 'Famílias'
    })
    .assign(grupo = lambda x: pd.Categorical(
        x['grupo'], 
        categories=['Pessoas', 'Famílias'])
    )
    .groupby(['ano', 'grupo', 'regiao', 'sigla_uf'])
    .agg('sum')
    .reset_index()
    .assign(
        qtde = lambda x: pd.to_numeric(
            x['qtde'], 
            downcast='integer'
        ),
        qtde_1e6 = lambda x: x['qtde'] / 1e6
    )
)

In [ ]:
df_qtde_por_ano_regiao_uf.head()

In [ ]:
df_qtde_por_ano_regiao_uf_grupo_pessoas = df_qtde_por_ano_regiao_uf.query('grupo == "Pessoas"')

In [ ]:
df_qtde_por_ano_regiao_uf_grupo_pessoas.head()

In [ ]:
for (regiao, df_por_regiao) in df_qtde_por_ano_regiao_uf_grupo_pessoas.groupby('regiao'):
    print(regiao)

    fig, ax = plt.subplots(
        figsize=(10, 3.5),
        dpi=300,
        layout='constrained'
    )

    sns.lineplot(
        df_por_regiao,
        x='ano',
        y='qtde_1e6',
        hue='sigla_uf',
        style='sigla_uf',
        markers=True,
        errorbar=None,
        dashes=False,
        ax=ax
    )

    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylabel('Quantidade (Milhões)')
    ax.set_xlabel('Ano')
    ax.set_xticks(anos, labels=anos, rotation=90)
    ax.grid(color='grey', linestyle='--', alpha=0.5)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1), ncols=9, frameon=False)

    fig.savefig(f'pessoas_cadunico_ufs_{regiao.lower()}', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
df_qtde_por_ano_regiao_uf_grupo_familias = df_qtde_por_ano_regiao_uf.query('grupo == "Famílias"')

In [ ]:
df_qtde_por_ano_regiao_uf_grupo_familias.head()

In [ ]:
for (regiao, df_por_regiao) in df_qtde_por_ano_regiao_uf_grupo_familias.groupby('regiao'):
    print(regiao)

    fig, ax = plt.subplots(
        figsize=(10, 3.5),
        dpi=300,
        layout='constrained'
    )

    sns.lineplot(
        df_por_regiao,
        x='ano',
        y='qtde_1e6',
        hue='sigla_uf',
        style='sigla_uf',
        markers=True,
        errorbar=None,
        dashes=False,
        ax=ax
    )

    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylabel('Quantidade (Milhões)')
    ax.set_xlabel('Ano')
    ax.set_xticks(anos, labels=anos, rotation=90)
    ax.grid(color='grey', linestyle='--', alpha=0.5)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1), ncols=9, frameon=False)

    fig.savefig(f'familias_cadunico_ufs_{regiao.lower()}', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
projecao_populacao = '../../compartilhado/projecoes_2024_tab1_idade_simples.ods'

In [ ]:
pl.read_ods(projecao_populacao)